# Pandas — czyszczenie danych

**Programowanie w Pythonie II** | Wykład 7
**Politechnika Opolska** | Analityka danych w biznesie

---

## Kontekst

Na W05 i W06 poznaliśmy Pandas — tworzenie DataFrame, selekcję (`loc`, `iloc`), filtrowanie, sortowanie. Ale pracowaliśmy na **czystych** danych. W praktyce dane z systemów ERP, eksportów z Excela, ankiet online czy baz SQL są **brudne**: brakujące wartości, duplikaty, błędne typy, niespójne nazwy.

Dzisiejszy wykład to **kompletny przewodnik czyszczenia danych** — najważniejsza umiejętność analityka.

```mermaid
graph TD
    A["Brudne dane\n(CSV, Excel, API)"] --> B["1. Diagnoza\ninfo, isna, describe"]
    B --> C["2. Braki NaN\nfillna, dropna"]
    C --> D["3. Duplikaty\ndrop_duplicates"]
    D --> E["4. Typy danych\nto_numeric, to_datetime"]
    E --> F["5. Tekst\nstr.strip, str.title"]
    F --> G["Czyste dane → Analiza"]
```

## Po tym wykładzie potrafisz:

1. **Diagnozować** jakość danych metodami `info()`, `isna()`, `describe()` (Bloom 2)
2. **Stosować** strategie uzupełniania braków: `fillna()` ze średnią, medianą lub `ffill()` — z uzasadnieniem wyboru (Bloom 3)
3. **Wykrywać i usuwać** duplikaty metodami `duplicated()`, `drop_duplicates()` z parametrem `subset` (Bloom 3)
4. **Konwertować** typy danych: `pd.to_numeric()`, `pd.to_datetime()` z `errors='coerce'` (Bloom 3)
5. **Normalizować** kolumny tekstowe operacjami `str.strip()`, `str.title()`, `str.replace()` (Bloom 3)
6. **Budować** kompletny pipeline czyszczenia od brudnych danych do gotowych do analizy (Bloom 4)

## Import bibliotek

Zawsze importujemy Pandas jako `pd` i NumPy jako `np` — to konwencja w całym ekosystemie data science.

In [ ]:
import pandas as pd
import numpy as np

print(f"Pandas: {pd.__version__}")
print(f"NumPy: {np.__version__}")

---

## 1. Dane do pracy — zamówienia sieci kawiarni

Pracujemy z danymi sieci kawiarni — 30 zamówień z małej sieci kawiarni z 4 filiami w Warszawie. Dataset zawiera typowe problemy spotykane w prawdziwych danych eksportowanych z systemu sprzedażowego: brakujące wartości, duplikaty, ceny jako tekst, niespójne nazwy napojów, rozmiarów i filii.

In [ ]:
# Zamowienia sieci kawiarni — brudne dane z systemu sprzedazowego
data = {
    'id_zamowienia': [
        1001, 1002, 1003, 1004, 1005, 1006, 1007, 1008, 1009, 1010,
        1011, 1012, 1013, 1014, 1015, 1016, 1017, 1018, 1019, 1020,
        1003, 1007, 1015,                          # duplikaty!
        1021, 1022, 1023, 1024, 1025, 1026, 1027
    ],
    'napoj': [
        'Latte', 'Espresso', 'Cappuccino', 'Americano', 'Flat White',
        'Mocha', 'Espresso', 'Latte', 'Matcha Latte', 'Americano',
        'Cappuccino', 'latte', 'Espresso', 'Mocha', 'Americano',
        'Flat White', 'LATTE', 'Cappuccino', 'Matcha Latte', 'Espresso',
        'Cappuccino', 'Espresso', 'Americano',
        'Mocha', '  FLAT WHITE  ', 'cappuccino', 'LATTE', 'Americano',
        'espresso', 'Matcha Latte'
    ],
    'rozmiar': [
        'M', 'S', 'L', 'M', 'L',
        'M', 'S', 'L', 'M', 'S',
        'l', 's', 'M', 'L', 'm',
        'S', 'l', 'M', 'brak', 'S',
        'L', 'S', 'm',
        'L', 'M', 's', 'M', 'l', 'S', 'L'
    ],
    'cena': [
        '14.00', '9.00', '13.00', '11.00', '15.00',
        '14.50', '9.00', '14.00', '16.00', '11.00',
        '13.00', '12.00', '9.00', 'brak', '11.00',
        '15.00', None, '13.00', '16.00', '9.00',
        '13.00', '9.00', '11.00',
        '14.50', '15.00', '13.00', '14.00', 'N/A', '9.00', '16.00'
    ],
    'ilosc': [
        1, 2, 3, 1, 2, 2, 5, 1, 2, 4,
        1, 3, 1, 2, 1, 8, 1, 1, 2, 1,
        3, 5, 1,
        1, 1, 2, None, 1, 3, 1
    ],
    'data_zamowienia': [
        '2024-01-15', '2024-01-16', '2024-01-16', '2024-01-17', '2024-01-18',
        '2024-01-18', '2024-01-19', '2024-01-20', '2024-01-20', '2024-01-21',
        '2024-01-22', '2024-01-22', '2024-01-23', '2024-01-24', '2024-01-25',
        '2024-01-25', '2024-01-26', '2024-01-27', None, '2024-01-28',
        '2024-01-16', '2024-01-19', '2024-01-25',
        '2024-01-29', '2024-01-29', '2024-01-30', '2024-01-30', '2024-01-31',
        None, '2024-02-01'
    ],
    'filia': [
        'Centrum', 'Mokotow', 'Centrum', 'Zoliborz', 'mokotow',
        'CENTRUM', 'Srodmiescie', 'centrum', 'Mokotow', 'Zoliborz',
        'Srodmiescie', 'mokotow', 'Zoliborz', None, 'Centrum',
        'srodmiescie', 'Mokotow', 'Centrum', 'Mokotow', 'ZOLIBORZ',
        'Centrum', 'Srodmiescie', 'Centrum',
        'Mokotow', 'Centrum', 'Zoliborz', 'zoliborz', 'Srodmiescie', 'centrum', 'Centrum'
    ]
}

df = pd.DataFrame(data)
print(f"Wymiary: {df.shape}")
print(f"Kolumny: {df.columns.tolist()}")

In [ ]:
# Podglad pierwszych 10 wierszy
df.head(10)

In [ ]:
# info() — pierwsza diagnoza: typy danych i liczba non-null
df.info()

`info()` pokazuje typ każdej kolumny i liczbę wartości non-null. To pierwsza komenda diagnostyczna — uruchamiamy ją **zawsze** po wczytaniu danych.

In [ ]:
# Liczba brakujacych wartosci (NaN) per kolumna
print("Brakujace wartosci:")
print(df.isna().sum())

print(f"\nProcent brakow:")
print((df.isna().sum() / len(df) * 100).round(1))

In [ ]:
# describe() — statystyki opisowe
# Uwaga: cena nie pojawi sie — Pandas traktuje ja jako tekst!
df.describe()

### Co widzimy?

| Problem | Szczegóły |
|---------|-----------|
| `cena` jest typu `object` (tekst) | Nie można policzyć średniej ani sumy |
| `data_zamowienia` jest typu `object` | Nie można filtrować po datach |
| `ilosc` jest typu `float64` | Powinna być `int`, ale NaN wymusza float |
| Braki (NaN) | cena: 1, ilosc: 1, data: 2, filia: 1 |
| Ukryte braki | `'brak'` i `'N/A'` w cenie — `isna()` ich **nie wykryje** |

### Analogia do Excela

W Excelu sprawdzamy braki ręcznie: `Ctrl+G` → Specjalne → Puste komórki. Albo wpisujemy `=CZY.BRAK(A1)` i kopiujemy w dół. W Pandas robimy to **jedną linią**: `df.isna().sum()`.

Ale uwaga: jeśli ktoś wpisał w Excelu *brak* zamiast zostawić pustą komórkę — Excel też tego nie wykryje jako brak. Dokładnie tak samo działa Pandas.

---

## 2. Brakujące wartości (NaN)

Brakujące dane to najczęstszy problem w analizie danych. Skąd się biorą?
- Klient nie wypełnił pola w formularzu
- Błąd eksportu z bazy danych
- Sensor nie wysłał odczytu
- Łączenie tabel z różnym zakresem dat

Pandas reprezentuje braki jako `NaN` (Not a Number) lub `NaT` (Not a Time) dla dat. Python `None` też staje się `NaN` w DataFrame.

In [ ]:
# isna() zwraca DataFrame True/False — mape brakow
print("Mapa brakow (pierwsze 5 wierszy):")
print(df.isna().head(5))

In [ ]:
# Wiersze z PRZYNAJMNIEJ JEDNYM brakiem
wiersze_z_nan = df[df.isna().any(axis=1)]
print(f"Wiersze z brakiem: {len(wiersze_z_nan)}")
wiersze_z_nan

In [ ]:
# isna() vs isnull() — to identyczne metody (aliasy)
print("isna() == isnull():", df.isna().equals(df.isnull()))

# notna() — odwrotnosc isna()
print("\nWiersze z kompletna cena:", df['cena'].notna().sum())
print("Uwaga: 'brak' i 'N/A' to NIE jest NaN — to zwykly tekst!")

### dropna() — usuwanie wierszy z brakami

`dropna()` usuwa wiersze zawierające NaN. Trzy warianty:

| Parametr | Działanie | Przykład |
|----------|-----------|----------|
| `how='any'` (domyślny) | Usuń jeśli jakikolwiek NaN | `df.dropna()` |
| `subset=['kol']` | Usuń jeśli NaN w konkretnej kolumnie | `df.dropna(subset=['filia'])` |
| `thresh=N` | Zostaw jeśli co najmniej N wartości non-null | `df.dropna(thresh=5)` |

In [ ]:
# dropna() — rozne warianty
print(f"Przed dropna: {len(df)} wierszy")

# Domyslnie: usun wiersz jesli MA JAKIKOLWIEK NaN
df_drop_all = df.dropna()
print(f"Po dropna():                        {len(df_drop_all)} wierszy")

# subset= — usun tylko jesli NaN w konkretnych kolumnach
df_drop_filia = df.dropna(subset=['filia'])
print(f"Po dropna(subset=['filia']):        {len(df_drop_filia)} wierszy")

# thresh= — zostaw wiersz jesli ma co najmniej 6 non-null (z 7 kolumn)
df_drop_thresh = df.dropna(thresh=6)
print(f"Po dropna(thresh=6):                {len(df_drop_thresh)} wierszy")

### fillna() — uzupełnianie braków

Zamiast usuwać wiersze (tracimy dane!), możemy uzupełnić braki. Cztery strategie:

| Strategia | Kiedy stosować | Przykład |
|-----------|----------------|----------|
| Stała wartość | Brak = *nie dotyczy* | `fillna(0)` lub `fillna('brak danych')` |
| Średnia | Dane symetryczne, bez outlierów | `fillna(df['kol'].mean())` |
| Mediana | Dane z wartościami odstającymi | `fillna(df['kol'].median())` |
| Forward fill | Szeregi czasowe, dane chronologiczne | `ffill()` |

In [ ]:
# Strategia 1: stala wartosc — uzupelniamy filie
df_s1 = df.copy()
df_s1['filia'] = df_s1['filia'].fillna('nieznana')
print("Wiersz 13 (wczesniej NaN):")
print(df_s1.loc[13, ['id_zamowienia', 'napoj', 'filia']])

In [ ]:
# Strategia 2: srednia vs Strategia 3: mediana
# Na kolumnie ilosc — jedyny NaN w kolumnie numerycznej
print(f"NaN w ilosc: {df['ilosc'].isna().sum()}")
print(f"Srednia ilosc: {df['ilosc'].mean():.2f}")
print(f"Mediana ilosc: {df['ilosc'].median():.1f}")

# Uzupelniamy mediana (odporna na wartosci odstajace)
df_s3 = df.copy()
df_s3['ilosc'] = df_s3['ilosc'].fillna(df_s3['ilosc'].median())
print(f"\nPo fillna(mediana): NaN = {df_s3['ilosc'].isna().sum()}")

In [ ]:
# Strategia 4: ffill() — forward fill (z poprzedniego wiersza)
# Przydatne w szeregach czasowych: brak odczytu? Wez poprzedni
df_s4 = df.copy()
df_s4['filia'] = df_s4['filia'].ffill()
print("ffill() — wiersz 13:")
print(df_s4.loc[13, ['id_zamowienia', 'filia']])

# bfill() — backward fill (z nastepnego wiersza)
df_s4b = df.copy()
df_s4b['filia'] = df_s4b['filia'].bfill()
print(f"\nbfill() — wiersz 13:")
print(df_s4b.loc[13, ['id_zamowienia', 'filia']])

In [ ]:
# interpolate() — interpolacja liniowa (dla danych numerycznych)
# Szacuje brak na podstawie sasiadow
przyklad = pd.Series([10.0, np.nan, np.nan, 40.0, 50.0])
print("Przed interpolacja:", przyklad.tolist())
print("Po interpolate():  ", przyklad.interpolate().tolist())
# NaN miedzy 10 a 40 -> 20, 30 (rownomierny rozklad)

### Zaawansowane: uzupełnianie braków per grupa (zapowiedź W08)

Czasem jedna mediana dla całej kolumny to za mało — lepsza jest mediana **per rozmiar**. Np. brakująca cena dużego napoju (L) powinna być uzupełniona medianą cen rozmiarów L, nie wszystkich napojów.

Ta technika używa `groupby().transform()` — metodę, którą dokładnie poznamy na W08. Tu pokazujemy ją jako zapowiedź.

In [ ]:
# fillna per grupa — groupby + transform (zapowiedz W08)
df_group = df.drop_duplicates().reset_index(drop=True).copy()
df_group['cena'] = pd.to_numeric(df_group['cena'], errors='coerce')

# Mediana ceny per rozmiar (po normalizacji nazw)
df_group['rozm_clean'] = df_group['rozmiar'].str.strip().str.upper()
mediana_per_rozm = df_group.groupby('rozm_clean')['cena'].transform('median')
df_group['cena'] = df_group['cena'].fillna(mediana_per_rozm)

# Jesli nadal sa NaN (rozmiar mial same NaN) — globalna mediana
df_group['cena'] = df_group['cena'].fillna(df_group['cena'].median())
print(f"NaN po fillna per grupa: {df_group['cena'].isna().sum()}")

### Kiedy usuwać, kiedy uzupełniać?

| Sytuacja | Decyzja | Dlaczego |
|----------|---------|----------|
| 80% kolumny to NaN | Usuń kolumnę (`drop(columns=...)`) | Kolumna bezużyteczna |
| 2% wierszy z NaN w kluczowej kolumnie | Usuń wiersze (`dropna(subset=...)`) | Mały koszt, dane wiarygodne |
| 10% braków w kolumnie numerycznej | Uzupełnij medianą | Mediana odporna na outliers |
| Dane czasowe, sporadyczne braki | `ffill()` | Kontynuacja trendu |
| Brak = *nie podano* (np. email) | `fillna('brak')` | Semantycznie poprawne |

> **Ciekawostka:** Według badań CrowdFlower/Appen (2016, potwierdzanych w kolejnych edycjach), analitycy danych spędzają **60–80% czasu na czyszczeniu danych**, a tylko 20–40% na właściwej analizie. Popularne powiedzenie w branży: *„80% data science to czyszczenie danych, 20% to narzekanie na czyszczenie danych."*

---

## 3. Duplikaty

Duplikaty to wiersze powtórzone w DataFrame — ten sam rekord zapisany dwa lub więcej razy. Typowe przyczyny: podwójny import z bazy, błąd w ETL, klient kliknął *Wyślij* dwa razy.

### Analogia do Excela

Excel ma wbudowaną funkcję: **Dane** → **Usuń duplikaty**. Pandas robi to samo: `drop_duplicates()`. Różnica: Pandas pozwala precyzyjnie kontrolować, które kolumny brać pod uwagę (`subset`) i które wystąpienie zostawić (`keep`).

In [ ]:
# duplicated() — wykrywanie duplikatow
# True = wiersz jest kopia wczesniejszego (domyslnie keep='first')
print(f"Liczba zduplikowanych wierszy: {df.duplicated().sum()}")

# Pokaz zduplikowane wiersze
print("\nZduplikowane wiersze:")
df[df.duplicated()]

In [ ]:
# keep=False — oznacza WSZYSTKIE wystapienia (oryginal + kopie)
print("Oryginaly i duplikaty razem:")
df[df.duplicated(keep=False)].sort_values('id_zamowienia')

In [ ]:
# Duplikaty po konkretnej kolumnie — subset
print(f"Duplikaty pelnych wierszy: {df.duplicated().sum()}")
print(f"Duplikaty wg id_zamowienia: {df.duplicated(subset=['id_zamowienia']).sum()}")

# Ile razy pojawia sie kazde id (tylko powtorzone)
print("\nPowtorzone id_zamowienia:")
counts = df['id_zamowienia'].value_counts()
print(counts[counts > 1])

In [ ]:
# drop_duplicates() — usuwanie duplikatow
przed = len(df)

# keep='first' (domyslnie) — zostaw pierwsze wystapienie
df_first = df.drop_duplicates()
print(f"Przed: {przed}, po drop_duplicates(): {len(df_first)}")

# keep='last' — zostaw ostatnie (przydatne: pozniejszy = aktualniejszy)
df_last = df.drop_duplicates(keep='last')
print(f"Po drop_duplicates(keep='last'): {len(df_last)}")

# subset= — duplikat tylko gdy te kolumny sie powtarzaja
df_by_id = df.drop_duplicates(subset=['id_zamowienia'])
print(f"Po drop_duplicates(subset=['id_zamowienia']): {len(df_by_id)}")

In [ ]:
# Po usunieciu wierszy — indeks ma 'dziury'
df_clean = df.drop_duplicates()
print("Indeks po drop_duplicates:", df_clean.index.tolist())

# reset_index(drop=True) — czyste 0, 1, 2, ...
df_clean = df_clean.reset_index(drop=True)
print(f"\nPo reset_index: 0..{len(df_clean)-1}, shape: {df_clean.shape}")

**Podsumowanie dotychczasowe:** po krokach 2 i 3 mamy dane bez duplikatów (27 unikalnych zamówień) i wiemy, gdzie są braki NaN. Teraz musimy naprawić **typy danych** — cena i data są tekstem, a nie liczbami i datami.

---

## 4. Konwersja typów danych

Trzecia kategoria problemów: **złe typy danych**. Ceny zapisane jako tekst, daty jako string, liczby całkowite z NaN zmienione na float. Pandas nie może policzyć średniej ze stringów — musimy skonwertować typy.

In [ ]:
# Praca na danych bez duplikatow
df_work = df.drop_duplicates().reset_index(drop=True).copy()

print("Typy danych:")
print(df_work.dtypes)

In [ ]:
# Problem: cena to tekst — nie mozna liczyc
print(f"cena — typ: {df_work['cena'].dtype}")
print(f"Przyklad: {df_work['cena'].head(3).tolist()}")

try:
    print(f"Srednia cena: {df_work['cena'].mean()}")
except TypeError as e:
    print(f"\nBlad mean(): {type(e).__name__} — nie mozna liczyc sredniej ze stringow!")

In [ ]:
# Sprawdzmy co sie kryje w kolumnie cena
print("Nietypowe wartosci w cena:")
for val in df_work['cena'].unique():
    try:
        float(val)
    except (ValueError, TypeError):
        print(f"  '{val}' (typ: {type(val).__name__})")

In [ ]:
# astype(float) — prosta konwersja, ale wywali blad przy 'brak'
try:
    df_work['cena'].astype(float)
    print("astype(float) OK!")
except ValueError as e:
    print(f"Blad astype(float): {e}")

### pd.to_numeric() — bezpieczna konwersja na liczby

`pd.to_numeric()` z parametrem `errors='coerce'` to bezpieczna alternatywa: wartości, których nie da się skonwertować, zamienia na `NaN` zamiast rzucać wyjątek.

| Parametr `errors` | Działanie |
|--------------------|-----------|
| `'raise'` (domyślny) | Rzuca błąd przy niepoprawnej wartości |
| `'coerce'` | Niepoprawna wartość → NaN |
| `'ignore'` | Zwraca oryginalną kolumnę bez zmian |

In [ ]:
# pd.to_numeric(errors='coerce') — bezpieczna konwersja
df_work['cena'] = pd.to_numeric(df_work['cena'], errors='coerce')

print(f"Typ po konwersji: {df_work['cena'].dtype}")
print(f"NaN po konwersji: {df_work['cena'].isna().sum()}")
print(f"Srednia cena (bez NaN): {df_work['cena'].mean():.2f} zl")

In [ ]:
# Uzupelniamy braki mediana (odporna na outliers)
mediana_ceny = df_work['cena'].median()
print(f"Mediana ceny: {mediana_ceny:.2f} zl")

df_work['cena'] = df_work['cena'].fillna(mediana_ceny)
print(f"NaN po fillna: {df_work['cena'].isna().sum()}")
print(f"Srednia cena (z uzupelnieniami): {df_work['cena'].mean():.2f} zl")

In [ ]:
# Naprawa ilosc: NaN wymusil float64, a powinno byc int
# Po uzupelnieniu NaN mozemy zamienic na int
df_work['ilosc'] = df_work['ilosc'].fillna(df_work['ilosc'].median())
df_work['ilosc'] = df_work['ilosc'].astype(int)
print(f"ilosc — typ: {df_work['ilosc'].dtype}")
print(f"Przyklad: {df_work['ilosc'].head(5).tolist()}")

### pd.to_datetime() — konwersja dat

`pd.to_datetime()` konwertuje stringi na typ `datetime64`. Po konwersji możemy używać **accessora `.dt`** — wyciąganie roku, miesiąca, dnia tygodnia itd.

In [ ]:
# pd.to_datetime() z errors='coerce'
print(f"data_zamowienia — typ: {df_work['data_zamowienia'].dtype}")

df_work['data_zamowienia'] = pd.to_datetime(
    df_work['data_zamowienia'], errors='coerce'
)
print(f"Typ po konwersji: {df_work['data_zamowienia'].dtype}")
print(f"NaN po konwersji: {df_work['data_zamowienia'].isna().sum()}")

In [ ]:
# .dt accessor — wydobywanie elementow daty
df_work['rok'] = df_work['data_zamowienia'].dt.year
df_work['miesiac'] = df_work['data_zamowienia'].dt.month
df_work['dzien_tygodnia'] = df_work['data_zamowienia'].dt.day_name()

print("Daty z rozbitymi elementami:")
print(df_work[['data_zamowienia', 'rok', 'miesiac', 'dzien_tygodnia']].dropna().head(5))

In [ ]:
# Ile zamowien per dzien tygodnia?
print("Zamowienia per dzien tygodnia:")
print(df_work['dzien_tygodnia'].value_counts())

### Analogia do Excela

W Excelu konwersja typów często dzieje się automatycznie (Excel *zgaduje* typ). Czasem to powoduje problemy — numer PESEL `00412345678` zamieniony na liczbę traci wiodące zero. W Pandas mamy pełną kontrolę:

| Excel | Pandas | Uwagi |
|-------|--------|-------|
| `=WARTOŚĆ("123")` | `pd.to_numeric('123')` | Bezpieczniej z `errors='coerce'` |
| `=DATA(2024;1;15)` | `pd.to_datetime('2024-01-15')` | Rozpoznaje wiele formatów |
| `=CZY.BRAK(A1)` | `df['kol'].isna()` | Zwraca True/False |
| `=JEŻELI(CZY.BRAK(A1); 0; A1)` | `df['kol'].fillna(0)` | Jedna linia zamiast formuły |

In [ ]:
# astype('category') — typ kategoryczny (oszczednosc pamieci)
print(f"Unikalne rozmiary: {df_work['rozmiar'].nunique()}")

mem_object = df_work['rozmiar'].memory_usage(deep=True)
df_work['rozmiar_cat'] = df_work['rozmiar'].astype('category')
mem_cat = df_work['rozmiar_cat'].memory_usage(deep=True)

print(f"Pamiec object:   {mem_object} B")
print(f"Pamiec category: {mem_cat} B")
print(f"Oszczednosc: {(1 - mem_cat / mem_object) * 100:.0f}%")

> **Ciekawostka:** Nazwa *Pandas* nie pochodzi od zwierzęcia — to skrót od **Pan**el **Da**ta, terminu z ekonometrii oznaczającego dane obserwowane w wielu wymiarach (np. wiele firm w wielu latach). Twórca biblioteki, Wes McKinney, pracował w funduszu inwestycyjnym AQR Capital Management i potrzebował narzędzia do analizy danych finansowych. Pierwszą wersję napisał w 2008 roku — dziś Pandas to najpopularniejsza biblioteka do analizy danych w Pythonie, z ponad 40 000 gwiazdek na GitHubie.

---

## 5. Operacje na tekstach (accessor `str`)

Ostatnia kategoria problemów: **brudne teksty**. Dane z formularzy, eksportów, scrapingu — pełne niekonsekwentnych wielkości liter, spacji, literówek.

Pandas udostępnia accessor `.str` — każda metoda stringowa Pythona dostępna dla całej kolumny naraz, bez pętli `for`.

In [ ]:
# Ile unikalnych filii PRZED czyszczeniem?
print("Filie (unikalne) — przed czyszczeniem:")
print(sorted(df['filia'].dropna().unique()))
print(f"Liczba unikalnych: {df['filia'].nunique()}")

In [ ]:
# str.lower(), str.upper(), str.title()
print("str.lower():", df['filia'].str.lower().dropna().unique()[:5].tolist())
print("str.upper():", df['filia'].str.upper().dropna().unique()[:5].tolist())
print("str.title():", df['filia'].str.title().dropna().unique()[:5].tolist())

In [ ]:
# str.title() — normalizacja wielkosci liter
filie_clean = df['filia'].str.title()
print(f"Po str.title(): {sorted(filie_clean.dropna().unique())}")
print(f"Unikalne: {filie_clean.nunique()}")

In [ ]:
# str.strip() — usuwanie spacji z poczatku i konca
print("Napoje ze spacjami (przed strip):")
for p in df['napoj']:
    if p != p.strip():
        print(f"  '{p}'")

napoje_clean = df['napoj'].str.strip()
print(f"\nPo strip: '{napoje_clean.iloc[24]}'")

In [ ]:
# str.replace() — zamiana fragmentow tekstu
# Przyklad: normalizacja rozmiarow
rozm_clean = df['rozmiar'].str.strip().str.upper()
print("Rozmiary po strip().upper():")
print(sorted(rozm_clean.unique()))

`str.contains()`, `str.startswith()`, `str.endswith()` — metody filtrujące. Zwracają `True/False` dla każdego elementu, więc można ich używać jako maski do filtrowania DataFrame.

In [ ]:
# str.contains() — filtrowanie po zawartosci tekstu
# Znajdz wszystkie zamowienia z 'Latte' w nazwie napoju
latte = df[df['napoj'].str.contains('Latte', case=False, na=False)]
print(f"Napoje z 'Latte' w nazwie ({len(latte)}):")
print(latte[['id_zamowienia', 'napoj', 'cena']])

In [ ]:
# str.startswith() i str.endswith() — poczatek / koniec tekstu
print("Napoje zaczynajace sie na 'E':")
starts_e = df[df['napoj'].str.strip().str.startswith('E', na=False)]
print(starts_e['napoj'].unique())

In [ ]:
# str.len() — dlugosc tekstu (przydatne do walidacji danych)
print("Dlugosci nazw napojow:")
print(df['napoj'].str.len().describe().round(1))

# Bardzo krotkie nazwy — mogly byc obciete przy eksporcie?
krotkie = df[df['napoj'].str.strip().str.len() < 6]
print(f"\nNapoje z nazwa < 6 znakow ({len(krotkie)}):")
print(krotkie['napoj'].str.strip().unique())

### Kompletne czyszczenie kolumn tekstowych

Typowy wzorzec: `str.strip()` → `str.title()` (lub `str.lower()`) → opcjonalnie `str.replace()` dla wyjątków.

Ważna zasada: **czyszczenie tekstów robimy PRZED grupowaniem i analizą** — inaczej 'Centrum', 'centrum' i 'CENTRUM' to trzy różne filie.

In [ ]:
# Pelne czyszczenie tekstow na swiezej kopii
df_text = df.drop_duplicates().reset_index(drop=True).copy()

# Napoje: strip + title
df_text['napoj'] = df_text['napoj'].str.strip().str.title()
print(f"Napoje (unikalne): {df_text['napoj'].nunique()}")

# Rozmiary: strip + upper (S, M, L)
df_text['rozmiar'] = df_text['rozmiar'].str.strip().str.upper()
print(f"Rozmiary ({df_text['rozmiar'].nunique()}): "
      f"{sorted(df_text['rozmiar'].unique())}")

# Filie: strip + title
df_text['filia'] = df_text['filia'].str.strip().str.title()
print(f"Filie ({df_text['filia'].dropna().nunique()}): "
      f"{sorted(df_text['filia'].dropna().unique())}")

### Podsumowanie metod `str`

| Metoda | Działanie | Przykład |
|--------|-----------|----------|
| `str.strip()` | Usuwa spacje z początku/końca | `'  abc  '` → `'abc'` |
| `str.lower()` | Wszystkie małe | `'ABC'` → `'abc'` |
| `str.upper()` | Wszystkie wielkie | `'abc'` → `'ABC'` |
| `str.title()` | Pierwsza wielka | `'abc def'` → `'Abc Def'` |
| `str.replace()` | Zamiana tekstu | `'HR dept'` → `'HR'` |
| `str.contains()` | Czy zawiera? (True/False) | Filtrowanie wierszy |
| `str.startswith()` | Czy zaczyna się od? | Filtrowanie prefiksem |
| `str.len()` | Długość tekstu | Walidacja danych |

---

## 6. Kompletny pipeline czyszczenia

Teraz łączymy wszystkie kroki w jeden pipeline — od brudnych danych do gotowych do analizy. Kolejność ma znaczenie:

1. **Duplikaty** — usuwamy najpierw, żeby nie zaburzać statystyk
2. **Teksty** — normalizujemy zanim używamy do grupowania
3. **Typy** — konwertujemy po oczyszczeniu tekstów (żeby 'brak' nie przeszło)
4. **Braki** — uzupełniamy na końcu, gdy typy są już poprawne

In [ ]:
# ========================================
# KOMPLETNY PIPELINE CZYSZCZENIA
# ========================================
df_pipeline = pd.DataFrame(data)   # swieza kopia
print(f"START: {df_pipeline.shape}")

# Krok 1: Usun duplikaty
df_pipeline = df_pipeline.drop_duplicates().reset_index(drop=True)
print(f"Krok 1 — po dedup: {df_pipeline.shape}")

# Krok 2: Wyczysc teksty
df_pipeline['napoj'] = df_pipeline['napoj'].str.strip().str.title()
df_pipeline['rozmiar'] = df_pipeline['rozmiar'].str.strip().str.upper()
df_pipeline['filia'] = df_pipeline['filia'].str.strip().str.title()
print(f"Krok 2 — filie: {sorted(df_pipeline['filia'].dropna().unique())}")

# Krok 3: Konwersja typow
df_pipeline['cena'] = pd.to_numeric(df_pipeline['cena'], errors='coerce')
df_pipeline['data_zamowienia'] = pd.to_datetime(
    df_pipeline['data_zamowienia'], errors='coerce'
)
print(f"Krok 3 — cena: {df_pipeline['cena'].dtype}, "
      f"data: {df_pipeline['data_zamowienia'].dtype}")

# Krok 4: Uzupelnij braki
med_cena = df_pipeline['cena'].median()
df_pipeline['cena'] = df_pipeline['cena'].fillna(med_cena)
df_pipeline['ilosc'] = df_pipeline['ilosc'].fillna(df_pipeline['ilosc'].median())
df_pipeline['filia'] = df_pipeline['filia'].fillna('Nieznana')
# data_zamowienia — zostawiamy NaN (nie da sie sensownie zgadnac daty zamowienia)
print(f"Krok 4 — NaN razem: {df_pipeline.isna().sum().sum()} "
      f"(pozostalo {df_pipeline['data_zamowienia'].isna().sum()} w dacie — "
      f"nie da sie zgadnac daty, zostawiamy)")

In [ ]:
# Weryfikacja koncowa
print("=== WERYFIKACJA ===")
print(f"Shape: {df_pipeline.shape}")
print(f"\nTypy danych:")
print(df_pipeline.dtypes)
print(f"\nBraki per kolumna:")
print(df_pipeline.isna().sum())

In [ ]:
# Analiza biznesowa na czystych danych
df_pipeline['wartosc'] = df_pipeline['cena'] * df_pipeline['ilosc']

print("=== RAPORT SPRZEDAZY KAWIARNI ===")
print(f"Laczna wartosc zamowien: {df_pipeline['wartosc'].sum():,.2f} zl")
print(f"Srednia wartosc zamowienia: {df_pipeline['wartosc'].mean():,.2f} zl")

print(f"\nTop 5 napojow wg wartosci:")
print(df_pipeline.groupby('napoj')['wartosc']
      .sum()
      .sort_values(ascending=False)
      .head(5)
      .round(2))

print(f"\nZamowienia per filia:")
print(df_pipeline['filia'].value_counts())

Ta analiza byłaby **niemożliwa** na brudnych danych: 'Centrum', 'centrum' i 'CENTRUM' to dla Pandas trzy różne filie. Po czyszczeniu — jedna. Dlatego pipeline czyszczenia to **fundament** każdej analizy.

---

## Podsumowanie — ściąga

### Diagnoza

| Chcę... | Kod | Wynik |
|---------|-----|-------|
| Zobaczyć typy i braki | `df.info()` | Typy + non-null count |
| Policzyć braki | `df.isna().sum()` | NaN per kolumna |
| Procent braków | `(df.isna().sum() / len(df) * 100).round(1)` | % NaN |
| Statystyki numeryczne | `df.describe()` | mean, std, min, max... |

### Brakujące wartości

| Chcę... | Kod | Uwagi |
|---------|-----|-------|
| Usunąć wiersze z NaN | `df.dropna()` | Usuwa jeśli jakikolwiek NaN |
| Usunąć NaN w konkretnej kol. | `df.dropna(subset=['kol'])` | Precyzyjne |
| Uzupełnić stałą | `df['kol'].fillna(0)` | Brak = *nie dotyczy* |
| Uzupełnić średnią | `df['kol'].fillna(df['kol'].mean())` | Dane symetryczne |
| Uzupełnić medianą | `df['kol'].fillna(df['kol'].median())` | Odporna na outliers |
| Forward fill | `df['kol'].ffill()` | Szeregi czasowe |

### Duplikaty

| Chcę... | Kod | Uwagi |
|---------|-----|-------|
| Policzyć duplikaty | `df.duplicated().sum()` | Liczba |
| Zobaczyć duplikaty | `df[df.duplicated(keep=False)]` | Wszystkie wystąpienia |
| Usunąć duplikaty | `df.drop_duplicates()` | Zostaw pierwsze |
| Usunąć wg kolumny | `df.drop_duplicates(subset=['id'])` | Po konkretnej kol. |
| Zresetować indeks | `df.reset_index(drop=True)` | Po usunięciu wierszy |

### Konwersja typów

| Chcę... | Kod | Uwagi |
|---------|-----|-------|
| String na liczbę | `pd.to_numeric(df['kol'], errors='coerce')` | Bezpieczne |
| String na datę | `pd.to_datetime(df['kol'], errors='coerce')` | Bezpieczne |
| Rok z daty | `df['kol'].dt.year` | `.dt` accessor |
| Dzień tygodnia | `df['kol'].dt.day_name()` | Nazwa po angielsku |
| Typ category | `df['kol'].astype('category')` | Oszczędność pamięci |

### Czyszczenie tekstu

| Chcę... | Kod | Uwagi |
|---------|-----|-------|
| Usunąć spacje | `df['kol'].str.strip()` | Początek i koniec |
| Małe litery | `df['kol'].str.lower()` | Cały tekst |
| Title Case | `df['kol'].str.title()` | Pierwsze Wielkie |
| Zamienić tekst | `df['kol'].str.replace('a', 'b', regex=False)` | Find & replace |
| Filtrować po tekście | `df[df['kol'].str.contains('szukam', na=False)]` | Zawiera |

---

## Źródła i referencje

| Temat | Link |
|-------|------|
| Pandas — Working with missing data | https://pandas.pydata.org/docs/user_guide/missing_data.html |
| Pandas — `isna()` / `notna()` | https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.isna.html |
| Pandas — `fillna()` | https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.fillna.html |
| Pandas — `dropna()` | https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.dropna.html |
| Pandas — `duplicated()` | https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.duplicated.html |
| Pandas — `drop_duplicates()` | https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.drop_duplicates.html |
| Pandas — `pd.to_numeric()` | https://pandas.pydata.org/docs/reference/api/pandas.to_numeric.html |
| Pandas — `pd.to_datetime()` | https://pandas.pydata.org/docs/reference/api/pandas.to_datetime.html |
| Pandas — `astype()` | https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.astype.html |
| Pandas — Text (str accessor) | https://pandas.pydata.org/docs/user_guide/text.html |

---

## 7. Mini-ćwiczenia na wykładzie

### Ćwiczenie A — Diagnoza danych pogodowych

Dane z 10 stacji meteorologicznych. Zdiagnozuj problemy.

In [ ]:
# Cwiczenie A — dane pogodowe
pogoda = pd.DataFrame({
    'stacja': ['Warszawa', 'Krakow', 'Gdansk', 'Wroclaw', 'Poznan',
               'Lodz', 'Katowice', 'Szczecin', 'Lublin', 'Warszawa'],
    'temperatura': [5.2, None, 3.1, 4.8, None, 5.0, 3.9, None, 4.5, 5.2],
    'wilgotnosc': ['78%', '82%', 'brak', '75%', '80%', None, '85%', '79%', '77%', '78%'],
    'cisnienie': [1013, 1010, 1015, 1012, 1011, 1014, 1009, 1016, 1013, 1013]
})

# Zadanie:
# 1. Ile jest duplikatow?
# 2. Ile brakow (NaN) w kazdej kolumnie?
# 3. Jaki typ ma kolumna 'wilgotnosc' i dlaczego to problem?

In [ ]:
# Rozwiazanie A
print(f"1. Duplikaty: {pogoda.duplicated().sum()}")
print(f"\n2. Braki NaN:")
print(pogoda.isna().sum())
print(f"\n3. Typ wilgotnosc: {pogoda['wilgotnosc'].dtype}")
print("   Problem: tekst z '%' — nie mozna liczyc sredniej!")

### Ćwiczenie B — Czyszczenie nazw napojów

In [ ]:
# Cwiczenie B — brudne nazwy
napoje_b = pd.Series([
    '  Latte  ', 'latte', 'LATTE',
    'Cappuccino', 'cappuccino', 'CAPPUCCINO',
    'Flat White', '  flat white', 'FLAT WHITE  '
])

# Zadanie: wyczysc nazwy tak, aby zostaly 3 unikalne napoje
# Podpowiedz: strip() + title()

In [ ]:
# Rozwiazanie B
napoje_clean = napoje_b.str.strip().str.title()
print(f"Przed: {napoje_b.nunique()} unikalnych")
print(f"Po:    {napoje_clean.nunique()} unikalnych")
print(f"Wartosci: {napoje_clean.unique().tolist()}")

### Ćwiczenie C — Pipeline na danych sportowych

In [ ]:
# Cwiczenie C — wyniki maratonu
maraton = pd.DataFrame({
    'zawodnik': ['Jan Kowalski', 'jan kowalski', 'Anna Nowak',
                 'Piotr Wisniewski', 'ANNA NOWAK', 'Jan Kowalski'],
    'czas_min': ['215', '215', '198', 'brak', '198', '215'],
    'kategoria': ['M30', 'm30', 'K25', 'M40', 'k25', 'M30'],
    'miasto': ['Warszawa', 'warszawa', 'Krakow', 'Gdansk', 'krakow', 'Warszawa']
})

# Zadanie — pelny pipeline:
# 1. Usun duplikaty
# 2. Wyczysc teksty (title dla zawodnik/miasto, upper dla kategoria)
# 3. Usun duplikaty ponownie (po czyszczeniu moga byc nowe!)
# 4. Skonwertuj czas_min na liczbe (to_numeric)
# 5. Uzupelnij brak mediana
# Oczekiwany wynik: 4 wiersze, 0 NaN

In [ ]:
# Rozwiazanie C
m = maraton.copy()
m = m.drop_duplicates().reset_index(drop=True)             # 1
m['zawodnik'] = m['zawodnik'].str.strip().str.title()       # 2a
m['miasto'] = m['miasto'].str.strip().str.title()           # 2b
m['kategoria'] = m['kategoria'].str.strip().str.upper()     # 2c
m = m.drop_duplicates().reset_index(drop=True)              # 3 — nowe duplikaty po czyszczeniu!
m['czas_min'] = pd.to_numeric(m['czas_min'], errors='coerce')  # 4
m['czas_min'] = m['czas_min'].fillna(m['czas_min'].median())   # 5

print(f"Shape: {m.shape}")
print(f"NaN: {m.isna().sum().sum()}")
print(m)

---

## Podgląd laboratorium

Na laboratorium będziecie czyścić **dane działu HR** — inny dataset, te same techniki. Ćwiczenia krok po kroku.

**Start (3 komendy):**
```
cd C:\Users\student\python2
.venv\Scripts\Activate.ps1
code .
```

**Co będziecie robić:**
1. Diagnoza brudnego datasetu HR (30 pracowników, duplikaty, NaN, złe typy)
2. Usuwanie duplikatów i resetowanie indeksu
3. Konwersja wynagrodzeń ze stringa na float (`pd.to_numeric`)
4. Czyszczenie nazw działów (`str.strip().str.title()`)
5. Pełny pipeline: od brudnych danych do raportu per dział

---

## Na koniec

> *"It doesn't matter how beautiful your theory is, it doesn't matter how smart you are. If it doesn't agree with experiment, it's wrong."*
> — Richard Feynman, *The Character of Physical Law* (1965)
>
> *„Nie ma znaczenia, jak piękna jest twoja teoria ani jak mądry jesteś. Jeśli nie zgadza się z doświadczeniem — jest błędna."*

W analizie danych to oznacza: **analiza jest tak dobra, jak dane na wejściu**. Brudne dane = błędne wnioski. Czyszczenie to nie nudna robota — to fundament każdej wiarygodnej analizy.

**Za tydzień:** W08 — Pandas: łączenie i agregacja (`merge`, `groupby`, `pivot_table`). Nauczycie się łączyć DataFramy jak tabele w SQL i tworzyć raporty agregowane.